# Reporte de Auditoría y Cambios Sugeridos · Módulo de API y Servidor

### 1. Archivo: src/scoring.py
Nombre del error: Inferencia predictiva simulada (Mock Scoring Heuristics).

Explicación de lo que tenemos: La función score_transaction no está ejecutando el modelo XGBoost entrenado. En su lugar, utiliza un sistema de reglas heurísticas manuales basadas en condicionales if (evaluando si el tipo es TRANSFER o si el monto supera un percentil arbitrario), asignando incrementos de score fijos (+0.30, +0.20). Esto invalida el propósito de tener un modelo de Inteligencia Artificial en producción.

Explicación del cambio a realizar: Reemplazar las reglas manuales por la carga asíncrona del pipeline serializado (xgb_fraud_pipeline.joblib) mediante joblib, transformar los datos entrantes del esquema de Pydantic a un DataFrame de Pandas y calcular la probabilidad real con predict_proba().

Líneas de código que se modifican porque están mal (Bloque de evaluación heurística):

    # Lógica mock actual en src/scoring.py
    score = 0.10

    if tx.type in [TransactionType.TRANSFER, TransactionType.CASH_OUT]:
        score += 0.30

    if tx.amount > 50000:
        score += 0.20

    if tx.ip_country in ["US", "GB"]:
        score += 0.15


Líneas de código que estarían mejor:

    Python
    import joblib
    import pandas as pd
    import os

    # Carga única del artefacto real en el ámbito global del módulo
    MODEL_PATH = os.path.join(os.path.dirname(__file__), "..", "models", "xgb_fraud_pipeline.joblib")
    pipeline_real = joblib.load(MODEL_PATH)

    def score_transaction(tx: Transaction) -> tuple[float, RiskLevel]:
        # Convertimos el esquema Pydantic directamente al formato tabular que espera el pipeline
        input_data = pd.DataFrame([tx.model_dump()])

        # Inferencia matemática real del modelo XGBoost
        score_real = float(pipeline_real.predict_proba(input_data)[0][1])

        # Asignación de riesgo basada en el score real
        if score_real >= 0.75:
            return score_real, RiskLevel.HIGH
        elif score_real >= 0.45:
            return score_real, RiskLevel.MEDIUM
        return score_real, RiskLevel.LOW

### 2. Archivo: src/storage.py
Nombre del error: Volatilidad de estado por almacenamiento en memoria (In-Memory State Loss).

Explicación de lo que tenemos: El script gestiona las transacciones pendientes de revisión y el historial de feedback utilizando diccionarios y listas nativas de Python (_pending_transactions, _feedback_history). Debido a que hemos desplegado la API dentro de un contenedor Docker en AWS EC2, cualquier reinicio del contenedor, fallo del proceso de Uvicorn o escalado de servidores destruirá por completo la cola de trabajo de los analistas humanos.

Explicación del cambio a realizar: Aunque para el despliegue inmediato mantengamos la deuda técnica para dar servicio a Full Stack, se debe documentar la migración hacia una base de datos relacional (como PostgreSQL en AWS RDS) o una base de datos embebida ligera (sqlite3) con un volumen persistente mapeado en Docker.

Líneas de código que se modifican porque están mal (Estructuras volátiles globales):

    # Almacenamiento volátil en RAM dentro de src/storage.py
    _pending_transactions = {}
    _feedback_history = []

    def save_pending_transaction(tx_id: str, tx_data: dict):
        _pending_transactions[tx_id] = tx_data


Líneas de código que estarían mejor (Evolución persistente recomendada):

    import sqlite3

    DB_PATH = "data/sentinel_storage.db"

    def save_pending_transaction(tx_id: str, tx_data: dict):
        # Persistencia real en disco atómica para evitar pérdidas en AWS
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute(
            "INSERT OR REPLACE INTO pending_transactions (id, data) VALUES (?, ?)",
            (tx_id, str(tx_data))
        )
        conn.commit()
        conn.close()

### 3. Archivo: api/routes/fraud.py
Nombre del error: Desacoplamiento de umbrales operativos (Hardcoded Risk Gateways).

Explicación de lo que tenemos: Al evaluar la acción final (ALLOW, REVIEW, BLOCK), las rutas utilizan condicionales basados en valores estáticos duros introducidos directamente en el código (if score >= 0.7 o elif score >= 0.2). Esto rompe por completo el trabajo que hiciste en el notebook de Machine Learning, donde calculaste matemáticamente un best_threshold.joblib optimizado.

Explicación del cambio a realizar: Importar dinámicamente el umbral óptimo guardado durante la fase de modelado para sincronizar de verdad las fronteras de decisión de la API con los experimentos del Jupyter Notebook.

Líneas de código que se modifican porque están mal (Líneas de decisión estáticas):


    # Decisiones desalineadas con el modelo en api/routes/fraud.py
    if score >= 0.7:
        action = ActionType.BLOCK
    elif score >= 0.2:
        action = ActionType.REVIEW
    else:
        action = ActionType.ALLOW

Líneas de código que estarían mejor:

    import joblib
    import os

    THRESHOLD_PATH = os.path.join(os.path.dirname(__file__), "..", "..", "models", "best_threshold.joblib")
    # Si el archivo no existe por problemas de despliegue, usamos 0.80 como fallback seguro
    OPTIMAL_THRESHOLD = joblib.load(THRESHOLD_PATH) if os.path.exists(THRESHOLD_PATH) else 0.80

    # Sincronización matemática estricta entre Data Science y API
    if score >= OPTIMAL_THRESHOLD:
        action = ActionType.BLOCK
    elif score >= (OPTIMAL_THRESHOLD * 0.6): # Escala proporcional de revisión
        action = ActionType.REVIEW
    else:
        action = ActionType.ALLOW